In [ ]:
!pip install openai ragas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.8/358.8 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.7/84.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.9/484.9 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.4/226.4 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 117.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/9

In [ ]:
!pip install python-dotenv

In [ ]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextRecall, ContextPrecision, Faithfulness, AnswerRelevancy, AnswerAccuracy, ContextRelevance, ResponseGroundedness
from ragas.embeddings import HuggingFaceEmbeddings
import os
from dotenv import load_dotenv

In [ ]:
load_dotenv("/content/abc")
token = os.getenv("OPENAI_TOKEN")
print(token)
token = token

None


In [ ]:
# Setup LLM
client = AsyncOpenAI(api_key = token)
llm = llm_factory("gpt-4o-mini", client=client)
# Create Embedding
# embeddings = HuggingFaceEmbeddings(model="dangvantuan/vietnamese-document-embedding",
#     normalize_embeddings = True,
#     device = "cpu",
#     trust_remote_code = True)
# Create metric
# scorerCR = ContextRecall(llm=llm)
# scorerCP = ContextPrecision(llm=llm)
# scorerFFN = Faithfulness(llm=llm)
# scorerAR = AnswerRelevancy(llm=llm, embeddings=embeddings)
# scorerAA = AnswerAccuracy(llm=llm)
# scorerCR = ContextRelevance(llm=llm)
scorerRG = ResponseGroundedness(llm=llm)

In [ ]:
import csv
import json
import asyncio

# Cấu hình
SAVE_INTERVAL = 20
BATCH_SIZE = 20  # Số lượng xử lý song song cùng lúc (tương đương SAVE_INTERVAL để dễ lưu)
CONCURRENCY_LIMIT = 5 # Quan trọng: Giới hạn số request cùng lúc để tránh lỗi API

INPUT = "/content/drive/MyDrive/results-KG_v2.json"
OUTPUT = "/content/drive/MyDrive/testdata/metrics_results_RG_KGcontinue.csv"

# Tạo Semaphore để kiểm soát số lượng chạy song song
sem = asyncio.Semaphore(CONCURRENCY_LIMIT)

async def eval_item(item):
    # Dùng Semaphore để đảm bảo không quá 10 items chạy cùng lúc
    async with sem:
        try:
            # SỬA LỖI CÚ PHÁP: Thêm dấu phẩy
            score = await scorerRG.ascore(
                # user_input=item["user_input"],
                response=item["response"],  # <--- Đã thêm dấu phẩy
                # reference=item["reference"]
                retrieved_contexts = item["retrieved_contexts"]
            )
            return item["id"], score.value
        except Exception as e:
            # In lỗi ra để debug nếu cần
            # print(f"Error id {item.get('id')}: {e}")
            return item.get("id"), "ERROR"

def write_csv(rows, mode="a"):
    with open(OUTPUT, mode, newline="", encoding="utf-8") as f:
        csv.writer(f).writerows(rows)

async def main(data):
    # Khởi tạo file CSV
    write_csv([("id", "answer_relevancy")], mode="w")

    total_items = len(data)
    done = 0

    # Xử lý theo từng batch (lô) để lưu file
    for i in range(0, total_items, SAVE_INTERVAL):
        # Lấy ra 1 nhóm data (ví dụ 20 items)
        batch_items = data[i : i + SAVE_INTERVAL]

        # Tạo danh sách các task (công việc) nhưng chưa chờ (await) ngay
        tasks = [eval_item(item) for item in batch_items]

        # Chạy song song tất cả tasks trong batch này và chờ kết quả
        results = await asyncio.gather(*tasks)

        # Lưu kết quả
        write_csv(results)

        done += len(results)
        print(f"Done: {done}/{total_items}")

def load_json(path):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("JSON must be a list")
    return data

# Chạy chương trình
data = load_json(INPUT)
await main(data[320:])

Done: 20/424
Done: 40/424
Done: 60/424
Done: 80/424
Done: 100/424
Done: 120/424
Done: 140/424
Done: 160/424
Done: 180/424
Done: 200/424
Done: 220/424
Done: 240/424
Done: 260/424
Done: 280/424
Done: 300/424
Done: 320/424
Done: 340/424
Done: 360/424
Done: 380/424
Done: 400/424
Done: 420/424
Done: 424/424


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
